# Import, playback, and clipboard
Run one cell at a time in a disposable local JupyterLab project. The browser media controls need this notebook's live Python kernel. Each call returns immediately; wait for the notebook media row, then run its inspection cell. Choose only the generated sample PNG when the file picker opens. The copy and paste controls require your click or keyboard paste.

## Disposable exact media references

In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib, math, struct, wave
from PIL import Image

server_root = Path.cwd().resolve()  # Start JupyterLab in this disposable project.
nonce = uuid4().hex
image_path = server_root / f'playback-{nonce}.png'
tone_path = server_root / f'playback-{nonce}.wav'
Image.new('RGB', (2, 2), (36, 112, 195)).save(image_path)
with wave.open(str(tone_path), 'wb') as output:
    output.setnchannels(1); output.setsampwidth(2); output.setframerate(16000)
    output.writeframes(b''.join(struct.pack('<h', int(9000 * math.sin(2 * math.pi * 440 * i / 16000))) for i in range(32000)))
image_ref = {'path': image_path.name, 'sha256': hashlib.sha256(image_path.read_bytes()).hexdigest()}
tone_ref = {'path': tone_path.name, 'sha256': hashlib.sha256(tone_path.read_bytes()).hexdigest()}
image_ref, tone_ref


## choose_file

In [ ]:
from nbinlineai.tools import choose_file
selected = choose_file(accept='image/png')
selected  # In the visible chooser select only the generated playback-*.png sample.


In [ ]:
selected.status, selected.media, selected.error  # No file is saved unless save_to is requested.


## open_media

In [ ]:
from nbinlineai.tools import open_media
opened = open_media(tone_ref)
opened  # The owned preview uses the exact WAV bytes and hash.


In [ ]:
opened.status, opened.result, opened.error  # Record opened.result['preview_id'] for controls below.


## play_media

In [ ]:
from nbinlineai.tools import play_media
assert opened.status == 'completed', 'Wait for the preview, then rerun this cell.'
playing = play_media(opened.result['preview_id'])
playing  # Click Play in the visible panel if the browser requires activation.


In [ ]:
playing.status, playing.result, playing.error


## pause_media

In [ ]:
from nbinlineai.tools import pause_media
paused = pause_media(opened.result['preview_id'])
paused


In [ ]:
paused.status, paused.result, paused.error


## seek_media

In [ ]:
from nbinlineai.tools import seek_media
seeked = seek_media(opened.result['preview_id'], 0.5)
seeked


In [ ]:
seeked.status, seeked.result, seeked.error  # Actual position can differ slightly by codec.


## set_media_volume

In [ ]:
from nbinlineai.tools import set_media_volume
volume = set_media_volume(opened.result['preview_id'], 0.25)
volume


In [ ]:
volume.status, volume.result, volume.error  # Some platforms disallow programmatic volume changes.


## close_media

In [ ]:
from nbinlineai.tools import close_media
closed = close_media(opened.result['preview_id'])
closed  # Closing the preview keeps tone_path intact.


In [ ]:
closed.status, closed.result, tone_path.exists(), closed.error


## copy_text

In [ ]:
from nbinlineai.tools import copy_text
copied = copy_text('nbinlineai disposable playback sample')
copied  # Click the visible Copy button or use the browser copy shortcut.


In [ ]:
copied.status, copied.result, copied.error


## paste_content

In [ ]:
from nbinlineai.tools import paste_content
pasted = paste_content(accept='text')
pasted  # Click the visible paste box and paste the sample text.


In [ ]:
pasted.status, pasted.result, pasted.error  # Plain text remains bounded; no background clipboard read.


In [ ]:
assert tone_path.exists() and image_path.exists()
tone_path.unlink()  # Delete only these two generated sample files.
image_path.unlink()
print('Disposable playback samples removed.')
